# Data Exercise 2 – Wind Resource Assessment (Part 2)

---

## Background

You submitted your initial wind resource assessment for Mt. Tom to your manager at **Aeolien Industries**. They've reviewed it and have come back with new requests:

- Re-run the log-law wind profile calculation for a **new hub height of 100 m**
- Evaluate operational wind speeds for a **new turbine model** (the Gamesa G126-2.625 MW) with different cut-in and cut-out speeds
- Calculate the **power available from the wind** using a standard physics equation
- Generate a **wind power curve** showing power output across the full range of operational wind speeds

This notebook builds directly on Part 1. You'll reuse several concepts from that notebook — and this time, you'll learn a new and important programming skill: writing your own **functions**.

---

## A note on what's new in this notebook

In Part 1, you ran each calculation once — one hub height, one cut-in/cut-out pair, one set of statistics. In this assignment, the manager wants you to:

- Run the **same log-law equation** for a *different* hub height (100 m instead of 60 m)
- Run the **same operational percentage calculation** with *different* cut-in/cut-out speeds
- Run the **same power equation** for an *entire range* of wind speeds

When you find yourself needing to **repeat the same calculation with different inputs**, that's the signal to write a **function**. You'll learn exactly how and why in the sections below.

---

## Step 0 – Imports and reload the data

We start every notebook the same way: import our libraries and load the dataset. This should look familiar from Part 1.

**Your turn:** The import cell below is complete, but the data-loading cell has a blank for you to fill in — you did this exact step in Part 1.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

print("Libraries loaded!")

Libraries loaded!


In [ ]:
# URL to the excel file we'll be opening
url = "https://github.com/radryan1979/clim341/raw/refs/heads/main/Data/HW1_MtToms_data.xlsx"

# TODO: Load the Excel data file into a DataFrame called df
# Hint: use pd.read_excel() — same as Part 1, the url variable is already provided above
df = pd.read_excel(___)

# TODO: Create the 'Day' column by dividing 'Hour' by 24 — same as Part 1
# Hint: see Question 7 in notebook for week-8
df['Day'] = df['Hour'] / ___

# Preview
print(f"Loaded {len(df)} rows.")
df.head()

Loaded 4464 rows.


,sample,Hour,m/s,Day
0,1,0.166667,1.609340,0.006944
1,2,0.333333,1.162300,0.013889
2,3,0.500000,1.162300,0.020833
3,4,0.666667,0.849376,0.027778
4,5,0.833333,1.162300,0.034722


---

## Introducing Functions

Before jumping into the assignment questions, let's talk about one of the most important concepts in programming: **functions**.

### What is a function?

You've already been *using* functions throughout Part 1 — `df.mean()`, `np.log()`, `pd.read_excel()` are all functions. A function is a **named, reusable block of code** that takes some inputs, does something, and gives back an output.

Now you're going to **write your own**.

### Why write a function?

Look at the log-law equation from Part 1:

$$U_{\text{hub}} = U_{\text{station}} \times \frac{\ln(z_{\text{hub}} / z_0)}{\ln(z_{\text{station}} / z_0)}$$

In Part 1, we applied it once, for a single hub height (60 m). Now the manager wants the same calculation for **100 m** — and we'll need it again to generate the power curve. If we copy-paste the same equation two or three times in the notebook, we introduce problems:

- If we discover a mistake (e.g., the wrong value of $z_0$), we have to fix it in *multiple places*
- The code becomes harder to read
- It's easy to introduce subtle errors between copies

The programming principle here is called **DRY: Don't Repeat Yourself**. The solution is to write the equation *once*, as a function, and then *call* it wherever we need it.

### Anatomy of a Python function

```python
def function_name(input1, input2):        # 'def' starts the definition; inputs are called 'parameters'
    """A short description of what this function does."""   # optional but good practice
    result = input1 + input2              # the calculation (indented!)
    return result                         # 'return' sends the answer back to the caller
```

Key rules:
- The function body must be **indented** (4 spaces or one tab)
- `return` is what sends the output back — without it, the function returns `None`
- You **define** a function once, then **call** it as many times as you need:

```python
answer = function_name(3, 5)   # calls the function with input1=3, input2=5
# answer is now 8
```

---

## Part A – Log-law function

### Writing the function

Let's turn the log-law equation into a function. It needs three inputs:
- `u_station` — the measured wind speed (m/s), can be a single number or an entire pandas column
- `z_hub` — the target height we want to extrapolate to (m)
- `z_station` — the height of the sensor (m), default 2.5 m
- `z0` — the surface roughness length (m), which you chose in Part 1

We can set **default values** for parameters that usually stay the same, so callers don't have to type them every time unless they want to change them:

```python
def my_function(required_input, optional_input=default_value):
```

Run the cell below to define the function — you'll see that nothing appears to happen. That's correct! Defining a function just stores it in memory; it doesn't run the calculation yet.

In [4]:
def log_law(u_station, z_hub, z_station=2.5, z0=0.5):
    """
    Extrapolate wind speed from measurement height to hub height
    using the logarithmic wind profile (log law).

    Parameters
    ----------
    u_station : float or array-like
        Wind speed measured at the station (m/s).
    z_hub : float
        Target height to extrapolate to, e.g. turbine hub height (m).
    z_station : float, optional
        Height of the measurement sensor. Default is 2.5 m.
    z0 : float, optional
        Aerodynamic roughness length of the surface (m). Default is 0.5 m.

    Returns
    -------
    float or array-like
        Wind speed extrapolated to z_hub (m/s).
    """
    u_hub = u_station * (np.log(z_hub / z0) / np.log(z_station / z0))
    return u_hub

print("log_law() function defined and ready to use.")

log_law() function defined and ready to use.


### Testing the function on a single value

Before trusting a function with a full dataset, it's good practice to **test it on a simple case** where you can verify the answer by hand.

Let's check: if `u_station = 5.0 m/s`, `z_hub = 60 m`, `z_station = 2.5 m`, and `z0 = 0.5 m`, we'd expect:

$$U_{60} = 5.0 \times \frac{\ln(60/0.5)}{\ln(2.5/0.5)} = 5.0 \times \frac{\ln(120)}{\ln(5)} \approx 5.0 \times \frac{4.787}{1.609} \approx 14.88 \text{ m/s}$$

In [5]:
# Test on a single wind speed value
test_result = log_law(u_station=5.0, z_hub=60)
print(f"Test: log_law(5.0 m/s, z_hub=60 m) = {test_result:.2f} m/s")
print(f"Expected: ~14.88 m/s")

Test: log_law(5.0 m/s, z_hub=60 m) = 14.87 m/s
Expected: ~14.88 m/s


In [ ]:
# TODO: Repeat the code above and try it with a different
# wind speed at the ground u_station and a different hub height, z_hub
# but this time, change the roughness length by passing in a new value for
# z0
test_result = log_law(u_station=___, z_hub=___, z0=___)

# TODO: copy and past the two print statements from the code above, change the text
# to print out the values you passed into the log_law function.

### Applying the function to the full dataset

One of the most powerful features of this function is that it works on **an entire pandas column at once**, not just a single number. This is because `np.log()` is vectorized — it operates element-by-element on arrays automatically.

Below we calculate hub-height wind speeds for **both 60 m and 100 m** by calling the same function twice with different `z_hub` values. Notice how clean and readable this is compared to writing out the equation twice.

In [ ]:
# Apply the log-law function to the entire wind speed column
# Calling it once for 60 m (same as Part 1) and once for the new 100 m hub height
df['u_60m']  = log_law(df['m/s'], z_hub=60)

# TODO: run the code for the u_100m column, fill in the ___
df['___'] = log_law(df['m/s'], z_hub=___)

# Preview the updated DataFrame
print("First 5 rows with hub-height wind speeds:")
df[['Day', 'm/s', 'u_60m', 'u_100m']].head()

First 5 rows with hub-height wind speeds:


,Day,m/s,u_60m,u_100m
0,0.006944,1.609340,4.787200,5.297995
1,0.013889,1.162300,3.457419,3.826326
2,0.020833,1.162300,3.457419,3.826326
3,0.027778,0.849376,2.526584,2.796171
4,0.034722,1.162300,3.457419,3.826326


Notice that calling `log_law(df['m/s'], z_hub=100)` required **zero extra effort** compared to the 60 m case — the function handles all the math. If we later decide to change $z_0$, we only need to update the function definition in one place, and both columns will be correct next time we run the notebook.

---

## Question 1 – Plot wind speed at three heights

Now plot all three wind speed profiles — surface (2.5 m), 60 m, and 100 m — on the **same figure**. This lets you visually compare the wind resource at different heights.

**Your turn:** The first two lines are provided. Add the third line for `u_100m`, and fill in the axis labels and title.

You can find a list of the colors for matplotlib [here](https://matplotlib.org/stable/gallery/color/named_colors.html)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

# TODO: In line 7 and line 10 you can change the color (see list of colors link above)
# and change the linewidth to style your graph how you would like

# Surface wind speed
ax.plot(df['Day'], df['m/s'],    color='steelblue', linewidth=0.8, label='Surface (2.5 m)')

# 60 m hub height (from Part 1)
ax.plot(df['Day'], df['u_60m'],  color='tomato',    linewidth=0.8, label='Hub height (60 m)')

# TODO: Add the 100 m line — follow the pattern above
# Pick a different color (e.g. 'forestgreen' or 'goldenrod')
ax.plot(df['Day'], df['___'],    color='___',       linewidth=0.8, label='___')

# TODO: Add x axis, y axis, and labels and title
# Hint, refer to Question 7 in the week-8 notebook for help finishing the code below
ax.___
ax.___
ax.___

ax.legend()
plt.tight_layout()
plt.show()

**✏️ Your answer:**

> *What do you notice about the wind speed at 100 m compared to 60 m and the surface? Is the difference between 60 m and 100 m larger or smaller than the difference between the surface and 60 m? Why might that be?*

---

## Part B – Operational percentage function

In Part 1, you calculated the percentage of time the wind speed at hub height fell within the operational range (3–15 m/s). Now the manager wants you to **repeat this calculation with a different turbine** — the Gamesa G126-2.625 MW — which has a wider operational range: cut-in at 1 m/s, cut-out at 25 m/s.

Instead of copying and pasting the code from Part 1, let's wrap it in a function.

### Writing the function

This function needs:
- A pandas Series (column) of wind speed values
- A cut-in speed
- A cut-out speed

It should return the **percentage** of values within the operational range.

The function below is mostly written — but the core calculation (the boolean mask) is left for you to fill in. You wrote this exact logic in Part 1.

In [ ]:
def operational_percentage(wind_speeds, cut_in, cut_out):
    """
    Calculate the percentage of wind speed observations that fall within
    the turbine's operational range [cut_in, cut_out].

    Parameters
    ----------
    wind_speeds : pandas Series
        Array of wind speed values (m/s).
    cut_in : float
        Minimum operational wind speed (m/s).
    cut_out : float
        Maximum operational wind speed (m/s).

    Returns
    -------
    float
        Percentage of observations within the operational range (0–100).
    """
    total = len(wind_speeds)

    # TODO: Create a boolean mask for values between cut_in and cut_out
    # Hint: this is the same logic you used in Part 1 - Question 9, line 10 of the code
    # Remember: use & to combine two conditions, and wrap each in parentheses
    operational = (wind_speeds ___ cut_in) & (wind_speeds ___ cut_out)  # fill in the comparison operators

    n_operational = operational.sum()
    return (n_operational / total) * 100

print("operational_percentage() function defined.")

## Question 2 – Original turbine (Part 1 check)

Call the function using the **original** cut-in (3 m/s) and cut-out (15 m/s) speeds on the 60 m wind speeds. This should match your answer from Part 1 — a good way to verify the function is working correctly.

In [ ]:
# Original turbine parameters
pct_original = operational_percentage(df['u_60m'], cut_in=3, cut_out=15)

print(f"Original turbine (3–15 m/s) at 60 m:")
print(f"  Operational: {pct_original:.1f}% of the time")

## Question 4 – Gamesa G126-2.625 MW (new turbine)

Now call the **same function** with the new turbine's parameters: cut-in 1 m/s, cut-out 25 m/s, applied to the **100 m** wind speeds.

**Your turn:** Fill in the function call below.

In [ ]:
# Gamesa G126 parameters
# TODO: Call operational_percentage() with the new turbine's cut-in/cut-out speeds
# Use df['u_100m'] as the wind speed column
pct_gamesa = operational_percentage(___, cut_in=___, cut_out=___)

print(f"Gamesa G126-2.625 MW (1–25 m/s) at 100 m:")
print(f"  Operational: {pct_gamesa:.1f}% of the time")

print(f"\nComparison:")
print(f"  Original turbine:  {pct_original:.1f}%")
print(f"  Gamesa G126:       {pct_gamesa:.1f}%")
print(f"  Difference:       +{pct_gamesa - pct_original:.1f} percentage points")

**✏️ Your answer:**

> *Is the Gamesa G126 a better choice for this site in terms of operational time? What accounts for the difference — wider cut-in, wider cut-out, or both?*

---

## Part C – Wind power function

Now for the most physically interesting part: calculating how much **power** the wind can deliver.

### The physics

The power available from the wind is:

$$P = \frac{1}{2} \rho A U^3$$

Where:
- $P$ = power (Watts)
- $\rho$ = air density = **1.225 kg/m³** (standard sea-level value)
- $A$ = area swept by the rotor blades = $\pi r^2$ (m²)
- $U$ = wind speed (m/s)

Notice the **cubic** relationship with wind speed ($U^3$). This means that doubling the wind speed produces **eight times** the power. Wind speed is by far the most important factor in determining the wind resource at a site.

### The turbine

The **Gamesa G126-2.625 MW** has a rotor diameter of **126 m**, so the radius is 63 m.

$$A = \pi r^2 = \pi \times 63^2 \approx 12{,}469 \text{ m}^2$$

### Why use a function here?

The power equation needs to be evaluated for:
1. The **average wind speed** at 100 m (a single number) — Question 5
2. **Every wind speed from 1 to 25 m/s** in 0.5 m/s steps (~49 values) — Question 6

Writing this as a function means we type the equation once and call it twice — for 1 value and for 49 values — with no code changes between the two uses.

In [ ]:
def wind_power(u, rotor_diameter, rho=1.225):
    """
    Calculate the power available from the wind using P = 0.5 * rho * A * U^3.

    Parameters
    ----------
    u : float or array-like
        Wind speed (m/s). Can be a single value or an array.
    rotor_diameter : float
        Diameter of the rotor swept area (m).
    rho : float, optional
        Air density (kg/m³). Default is 1.225 kg/m³ (standard sea level).

    Returns
    -------
    float or array-like
        Available wind power in Watts (W).
    """
    r = rotor_diameter / 2          # radius = diameter / 2
    A = np.pi * r**2                # swept area (m²)
    P = 0.5 * rho * A * u**3        # power equation
    return P

# Quick sanity check: what is the swept area for the Gamesa G126?
Area_gamesa = np.pi * (126/2)**2
print(f"Gamesa G126 rotor diameter: 126 m")
print(f"Swept area: π × 63² = {Area_gamesa:,.0f} m²")
print("\nwind_power() function defined.")

## Question 5 – Power at the mean wind speed

Calculate the power available from the wind at the **mean hub-height wind speed** (100 m) for the Gamesa G126.

We'll report the result in **kilowatts (kW)** by dividing watts by 1000, and also compute the **energy produced in one hour** in kWh — a unit familiar from electricity bills.

**Note:** In the numbers 1,000 and 1,000,000 on lines 9 and 10, Python allows you to use an underscore (_) to act as a comma for making the number easier to read in the code. You can use either 1000 or 1_000, whichever makes it easier for you as the coder.

In [ ]:
# Mean wind speed at 100 m hub height
u_mean_100m = df['u_100m'].mean()
print(f"Mean wind speed at 100 m: {u_mean_100m:.2f} m/s")

# Calculate power using the function
P_watts = wind_power(u=u_mean_100m, rotor_diameter=126)

# Convert to more useful units
P_kW  = P_watts / 1_000           # Watts → kilowatts
P_MW  = P_watts / 1_000_000       # Watts → megawatts
E_kWh = P_kW * 1                  # energy in 1 hour = power × time (kW × 1 hr = kWh)

print(f"\n--- Power Output at Mean Wind Speed ---")
print(f"  Power:          {P_watts:>12,.0f} W")
print(f"  Power:          {P_kW:>12,.1f} kW")
print(f"  Power:          {P_MW:>12,.2f} MW")
print(f"  Energy per hour:{E_kWh:>12,.1f} kWh")

**✏️ Your answer:**

> *Report the power output in watts and kilowatts. How does this compare to the turbine's nameplate capacity of 2.625 MW? Keep in mind that the nameplate capacity is the turbine's *maximum* — not the power at the average wind speed.*

---

## Question 6 – The wind power curve

A **wind power curve** shows the power output of a turbine as a function of wind speed. This is one of the most important tools in wind energy analysis — turbine manufacturers publish these curves for every machine they sell.

We want to evaluate power at wind speeds from **1 to 25 m/s in 0.5 m/s steps**. Instead of calculating each speed one by one (49 separate calculations!), we can generate all the speeds at once using `np.arange()` and pass the entire array to `wind_power()` — which handles arrays just as easily as single numbers, because `np.pi` and the `**` operator are both vectorized.

### Generating a range of values with `np.arange()`

`np.arange(start, stop, step)` generates evenly-spaced values:

```python
np.arange(1, 25.5, 0.5)  # 1.0, 1.5, 2.0, ..., 25.0
#                  ^^^  stop is exclusive, so use 25.5 to include 25.0
```

**Your turn:** Generate the wind speed array and calculate the power curve, then plot it.

In [ ]:
# TODO: Generate an array of wind speeds from 1 to 25 m/s in 0.5 m/s steps
# Hint: np.arange(start, stop, step) — remember stop is exclusive!
u_range = np.arange(___, ___, ___)

print(f"Generated {len(u_range)} wind speed values.")
print(f"First few: {u_range[:5]}")
print(f"Last few:  {u_range[-5:]}")

### Understanding the output: array indexing

`u_range` is a **NumPy array** — an ordered sequence of numbers. Once you've
filled in `np.arange()`, it will hold 49 evenly-spaced wind speed values from
1.0 to 25.0 m/s.

The `print` statements use **index notation** to peek at the ends of the array
without printing all 49 values:

- `u_range[:5]` — the **first 5 elements** (index 0 through 4).
  The `:5` means "start from the beginning, stop before index 5."

- `u_range[-5:]` — the **last 5 elements**.
  Negative indices count from the *end* of the array, so `-5` means
  "5 positions from the end, go to the finish."

If your array is correct, the output should look something like:
```
First few: [1.  1.5  2.  2.5  3. ]
Last few:  [23.  23.5  24.  24.5  25. ]
```

This is a quick sanity check — if the first and last values look right,
the whole array is likely correct.

In [ ]:
# TODO: Call wind_power() on the entire u_range array at once
# Because numpy is vectorized, this applies the equation to all 49 values simultaneously
P_curve_W  = wind_power(u=___, rotor_diameter=126)
P_curve_kW = P_curve_W / 1000    # convert to kW for a more readable y-axis

print(f"Power at 1 m/s:  {P_curve_kW[0]:.1f} kW")
print(f"Power at 10 m/s: {P_curve_kW[np.where(u_range==10)[0][0]]:.1f} kW")
print(f"Power at 25 m/s: {P_curve_kW[-1]:.1f} kW")

### Understanding the output: vectorized functions and array lookups

**Vectorization**

Calling `wind_power(u=u_range, ...)` passes the entire 49-element array in one
shot. NumPy applies the power equation to every wind speed simultaneously,
returning a new 49-element array of power values — one result per input.
This is called **vectorization**, and it's one of the main reasons NumPy is
used for scientific computing: no `for` loop needed.

`P_curve_kW` ends up as an array where each element corresponds to the same
position in `u_range`:

```
u_range     = [ 1.0,  1.5,  2.0, ..., 25.0 ]   # wind speeds (m/s)
P_curve_kW  = [ x.x,  x.x,  x.x, ..., x.x ]   # power at each speed (kW)
```

**The three print statements**

- `P_curve_kW[0]` — power at the **first** wind speed (1 m/s), using a direct
  index.

- `P_curve_kW[-1]` — power at the **last** wind speed (25 m/s), using a
  negative index (same idea as the previous cell).

- `P_curve_kW[np.where(u_range==10)[0][0]]` — power at exactly **10 m/s**.
  This one is more involved:
  - `u_range == 10` creates a boolean array (`True` where the condition is met,
    `False` everywhere else).
  - `np.where(...)` returns the *indices* where the condition is `True`.
  - `[0][0]` unpacks the result to get a single integer index.
  - That index is then used to look up the corresponding value in `P_curve_kW`.

  In plain English: *"find the position of 10 m/s in `u_range`, then use that
  position to get the power value."*

In the code below, you'll need to add the x and y axis labels to the ax.___ along with the title.

You may also want to explore stylizing your plot by changing colors and the linestyle for the `ax.axvline` and `ax.axvspan`.

Line styles are listed [here](https://matplotlib.org/stable/gallery/lines_bars_and_markers/linestyles.html) and color are listed [here](https://matplotlib.org/stable/gallery/color/named_colors.html)

In [ ]:
# Plot the wind power curve
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(u_range, P_curve_kW,
        color='steelblue',
        linewidth=2.0,
        marker='o',
        markersize=4,
        label='Gamesa G126-2.625 MW')

# Mark the mean wind speed on the curve with a vertical dashed line
# axvline draws a vertical line across the full height of the plot
# TODO: try plotting with different colors and linestles
ax.axvline(x=u_mean_100m, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Mean wind speed ({u_mean_100m:.1f} m/s)')

# Shade the operational range
# TODO: trying different colors for shading
ax.axvspan(1, 25, alpha=0.08, color='green', label='Operational range (1–25 m/s)')

# TODO: Add axis labels and title
ax.___
ax.___
ax.___

ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**✏️ Your answer:**

> *Describe the shape of the power curve. Why does it have this shape — what does the cubic relationship between power and wind speed ($U^3$) mean in practice? Where on the curve does the mean wind speed fall?*

---

## Summary table

Run the cell below for a complete summary of the results from both Part 1 and Part 2.

In [ ]:
print("=" * 52)
print("  Mt. Tom Wind Resource – Full Summary")
print("=" * 52)
print("  Wind speeds (mean / max)")
print(f"    Surface  (2.5 m):  {df['m/s'].mean():.2f} / {df['m/s'].max():.2f} m/s")
print(f"    Hub ht.  (60 m):   {df['u_60m'].mean():.2f} / {df['u_60m'].max():.2f} m/s")
print(f"    Hub ht.  (100 m):  {df['u_100m'].mean():.2f} / {df['u_100m'].max():.2f} m/s")
print("-" * 52)
print("  Operational time")
print(f"    Original turbine  (3–15 m/s @ 60 m):   {pct_original:.1f}%")
print(f"    Gamesa G126       (1–25 m/s @ 100 m):  {pct_gamesa:.1f}%")
print("-" * 52)
print("  Power (Gamesa G126, mean wind speed)")
print(f"    Mean U at 100 m:  {u_mean_100m:.2f} m/s")
print(f"    Power output:     {P_kW:,.1f} kW  ({P_MW:.3f} MW)")
print(f"    Energy per hour:  {E_kWh:,.1f} kWh")
print("=" * 52)

---

## 🎉 You've completed the wind resource assessment!

In this notebook you:

- Wrote **two reusable functions** — `log_law()` and `operational_percentage()` — and saw how they made it easy to re-run calculations with different inputs without rewriting code
- Applied the log-law to a **new hub height (100 m)** with a single function call
- Evaluated the **Gamesa G126-2.625 MW** turbine's operational range and compared it to the original turbine
- Calculated the **available wind power** using $P = \frac{1}{2}\rho A U^3$
- Generated and plotted a **wind power curve** by passing an array of wind speeds to the function in one step
